In [30]:
import glob
import os
import pandas as pd
from pathlib import Path
import random
from collections import defaultdict
from urllib.parse import urlparse
from pathlib import PurePosixPath
import shutil

In [2]:
data_path = "..\splits\lunch_dinner\data_correct_lunch_dinner_visible.csv"
data = pd.read_csv(data_path)
data.shape

(699, 33)

In [3]:
data.columns

Index(['sub', 'Carb', 'Protein', 'Fat', 'Fiber', 'Image path', 'Meal Type',
       'Amount Consumed', 'AUC', 'Calories', 'iAUC', 'Baseline_Libre', 'Age',
       'Gender', 'BMI', 'A1c', 'HOMA', 'Insulin', 'TG', 'Cholesterol', 'HDL',
       'Non HDL', 'LDL', 'VLDL', 'CHO/HDL ratio', 'Fasting BG',
       'Correct Nutrition', 'full_image_path', 'PWrapped', 'LogitWrapped',
       'LogitVisible', 'GapWrappedMinusVisible', 'IsWrapped'],
      dtype='object')

In [4]:
data["Amount Consumed"].value_counts(dropna=False)

100.0    439
1.0      104
300.0     22
200.0     22
75.0      21
3.0       16
400.0     12
600.0     11
2.0       11
4.0        7
5.0        6
6.0        4
500.0      4
60.0       4
50.0       3
9.0        2
70.0       2
700.0      2
0.0        2
8.0        1
7.0        1
150.0      1
375.0      1
90.0       1
Name: Amount Consumed, dtype: int64

In [5]:
data.groupby("sub")["Amount Consumed"].describe()

,count,mean,std,min,25%,50%,75%,max
sub,,,,,,,,
1,13.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
2,18.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
3,18.0,81.666667,20.651164,50.0,60.0,95.0,100.00,100.0
4,28.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
5,18.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
6,19.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
7,9.0,122.222222,44.095855,100.0,100.0,100.0,100.00,200.0
8,17.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0
9,17.0,100.000000,0.000000,100.0,100.0,100.0,100.00,100.0


In [6]:
all_data_lunch = data[data["Meal Type"] == "Lunch"]
all_data_lunch.shape

(362, 33)

In [7]:
all_data_dinner = data[data["Meal Type"] == "Dinner"]
all_data_dinner.shape

(337, 33)

In [8]:
all_data_lunch.groupby("sub")["Amount Consumed"].describe()

,count,mean,std,min,25%,50%,75%,max
sub,,,,,,,,
1,6.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
2,8.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
3,10.0,67.000000,16.363917,50.0,60.00,60.0,70.00,100.0
4,10.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
5,8.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
6,9.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
7,6.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
8,9.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
9,10.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0


In [9]:
all_data_dinner.groupby("sub")["Amount Consumed"].describe()

,count,mean,std,min,25%,50%,75%,max
sub,,,,,,,,
1,7.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
2,10.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
3,8.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
4,18.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
5,10.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
6,10.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
7,3.0,166.666667,57.735027,100.0,150.00,200.0,200.00,200.0
8,8.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0
9,7.0,100.000000,0.000000,100.0,100.00,100.0,100.00,100.0


In [13]:
stats = all_data_dinner.groupby("sub")["Amount Consumed"].describe()
stats_not_100 = stats[(stats["mean"] != 100.0) & (stats["mean"] != 1.0)]
stats_not_100

,count,mean,std,min,25%,50%,75%,max
sub,,,,,,,,
7,3.0,166.666667,57.735027,100.0,150.00,200.0,200.00,200.0
18,9.0,1.888889,0.927961,1.0,1.00,2.0,3.00,3.0
19,10.0,2.000000,0.816497,1.0,1.25,2.0,2.75,3.0
20,7.0,4.571429,1.902379,3.0,3.00,4.0,5.50,8.0
21,7.0,3.000000,1.290994,1.0,2.50,3.0,3.50,5.0
22,8.0,3.375000,1.060660,2.0,2.75,3.5,4.00,5.0
23,12.0,4.250000,1.912875,1.0,2.75,4.5,6.00,7.0
30,4.0,6.500000,3.000000,3.0,4.50,7.0,9.00,9.0
31,3.0,133.333333,57.735027,100.0,100.00,100.0,150.00,200.0


In [11]:
stats_not_100["count"].sum()

191.0

In [12]:
all_data_dinner.shape[0] - stats_not_100["count"].sum()

146.0

In [21]:
all_data_lunch[(all_data_lunch["Amount Consumed"] != 100.0) & (all_data_lunch["Amount Consumed"] != 1.0)][["sub", "Amount Consumed", "Image path"]]

,sub,Amount Consumed,Image path
31,3,50.0,photos/00000005-PHOTO-2020-3-11-12-9-0.jpg
32,3,90.0,photos/00000011-PHOTO-2020-3-12-14-25-0.jpg
34,3,60.0,photos/00000017-PHOTO-2020-3-13-12-33-0.jpg
35,3,60.0,photos/00000023-PHOTO-2020-3-14-13-33-0.jpg
37,3,60.0,photos/00000030-PHOTO-2020-3-15-14-6-0.jpg
39,3,70.0,photos/00000039-PHOTO-2020-3-16-13-32-0.jpg
43,3,60.0,photos/00000052-PHOTO-2020-3-18-13-34-0.jpg
45,3,70.0,photos/00000059-PHOTO-2020-3-19-14-7-0.jpg
47,3,50.0,photos/00000066-PHOTO-2020-3-20-12-45-0.jpg
161,10,75.0,photos/00000024-PHOTO-2020-6-24-11-21-0.jpg


In [18]:
all_data_lunch[(all_data_lunch["Amount Consumed"] != 100.0) & (all_data_lunch["Amount Consumed"] != 1.0)][["sub"]].count()

sub    29
dtype: int64

In [20]:
all_data_lunch.shape[0] - 29

333

In [24]:
all_data_lunch.loc[43]

sub                                                                       3
Carb                                                                  160.0
Protein                                                               304.0
Fat                                                                   153.0
Fiber                                                                  26.0
Image path                      photos/00000052-PHOTO-2020-3-18-13-34-0.jpg
Meal Type                                                             Lunch
Amount Consumed                                                        60.0
AUC                                                                 13050.5
Calories                                                              585.0
iAUC                                                             288.563953
Baseline_Libre                                                        106.8
Age                                                                      59
Gender      

In [25]:
all_data_dinner[all_data_dinner["sub"] == 7]

,sub,Carb,Protein,Fat,Fiber,Image path,Meal Type,Amount Consumed,AUC,Calories,...,VLDL,CHO/HDL ratio,Fasting BG,Correct Nutrition,full_image_path,PWrapped,LogitWrapped,LogitVisible,GapWrappedMinusVisible,IsWrapped
115,7,96.0,160.0,36.0,0.0,photos/00000007-PHOTO-2023-11-3-19-26-0.jpg,Dinner,100.0,8580.0,352.0,...,18,3.0,108,True,..\..\CGMacros\CGMacros-007\photos/00000007-PH...,0.000290,45.072144,53.216843,-0.081447,False
116,7,196.0,160.0,36.0,0.0,photos/00000013-PHOTO-2023-11-4-19-16-0.jpg,Dinner,200.0,11615.0,442.0,...,18,3.0,108,True,..\..\CGMacros\CGMacros-007\photos/00000013-PH...,0.000461,42.615400,50.295980,-0.076806,False
118,7,116.0,116.0,117.0,0.0,photos/00000019-PHOTO-2023-11-5-19-18-0.jpg,Dinner,200.0,6927.0,341.0,...,18,3.0,108,True,..\..\CGMacros\CGMacros-007\photos/00000019-PH...,0.003520,41.471054,47.116695,-0.056456,False


In [26]:
all_data_dinner.loc[118]

sub                                                                       7
Carb                                                                  116.0
Protein                                                               116.0
Fat                                                                   117.0
Fiber                                                                   0.0
Image path                      photos/00000019-PHOTO-2023-11-5-19-18-0.jpg
Meal Type                                                            Dinner
Amount Consumed                                                       200.0
AUC                                                                  6927.0
Calories                                                              341.0
iAUC                                                             1210.21519
Baseline_Libre                                                    50.666667
Age                                                                      66
Gender      

In [27]:
data_filtered = data[(data["Amount Consumed"] == 100.0) | (data["Amount Consumed"] == 1.0)]

In [28]:
data_filtered.shape

(543, 33)

In [29]:
data_filtered.to_csv('../splits/lunch_dinner/data_all_lunch_dinner_consumed.csv', index=False)

In [43]:
data_filtered = pd.read_csv("../splits/lunch_dinner/data_all_lunch_dinner_consumed.csv")

In [44]:
base_out = Path(r"..\pics_consumed")

for idx, row in data_filtered.iterrows():
    src = Path(row["full_image_path"])

    meal_type = str(row["Meal Type"]).strip().lower()
    sub = str(row["sub"]).strip()

    # keep only lunch and dinner
    if meal_type not in ["lunch", "dinner"]:
        continue

    if not src.exists():
        print(f"Missing file: {src}")
        continue

    dst_folder = base_out / meal_type / sub
    dst_folder.mkdir(parents=True, exist_ok=True)

    dst = dst_folder / src.name

    shutil.copy2(src, dst)

In [47]:
# lunch: 22
# 10: 00000005-PHOTO-2020-6-21-11-47-0
# 12: 00000029-PHOTO-2023-3-1-14-23-0, 00000048-PHOTO-2023-3-4-15-1-0
# 14: 00000005-PHOTO-2023-5-9-11-51-0, 00000028-PHOTO-2023-5-12-12-38-0, 00000058-PHOTO-2023-5-15-11-42-0, 00000090-PHOTO-2023-5-18-12-25-0
# 22: 00000052-PHOTO-2021-4-1-12-35-0, 00000071-PHOTO-2021-4-3-12-29-0, 00000104-PHOTO-2021-4-6-12-20-0
# 23: 00000047-PHOTO-2021-5-10-13-25-0, 00000081-PHOTO-2021-5-14-13-26-0
# 28: 00000005-PHOTO-2023-11-28-12-40-0, 00000036-PHOTO-2023-12-3-12-23-0
# 32: 00000025-PHOTO-2022-1-6-12-47-0
# 33: 00000018-PHOTO-2022-4-17-19-18-0 - empty -> 00000017-PHOTO-2022-4-17-19-18-0
# 34: 00000089-PHOTO-2022-3-7-11-53-0 - half eaten
# 36: 00000010-PHOTO-2022-3-30-12-7-0
# 42: 00000017-PHOTO-2025-7-26-11-27-0
# 44: 00000022-PHOTO-2022-10-18-13-53-0 - empty
# 46: 00000017-PHOTO-2025-5-3-11-47-0 - empty
# 48: 00000087-PHOTO-2022-11-23-13-22-0

In [48]:
# dinner: 17
# 4: 00000053-PHOTO-2023-9-15-20-45-0 - icecream, 00000064-PHOTO-2023-9-16-22-39-0 - icecream, 00000075-PHOTO-2023-9-17-20-2-0 - black (burger)
# 10: 00000008-PHOTO-2020-6-21-18-43-0 - banana
# 11: dinner-PHOTO-2020-10-4-19-0-0 - text
# 14: 00000007-PHOTO-2023-5-9-19-1-0, 00000013-PHOTO-2023-5-10-17-40-0, 00000070-PHOTO-2023-5-16-20-2-0, 00000083-PHOTO-2023-5-17-19-46-0, 
#00000093-PHOTO-2023-5-18-19-11-0
# 17: 00000014-PHOTO-2023-10-11-22-35-0 - chips, 00000047-PHOTO-2023-10-14-18-47-0 - hand
# 26: 00000045-PHOTO-2021-3-31-17-10-0
# 34: 00000103-PHOTO-2022-3-8-17-59-0 - empty
# 43: 00000120-PHOTO-2025-10-21-18-4-0, 00000127-PHOTO-2025-10-21-23-0-0, 00000157-PHOTO-2025-10-23-22-34-0

In [49]:
543 - 22 - 17

504